# 範例二：爬取 TIOBE 程式語言排行榜

使用 `requests` 與 `BeautifulSoup` 取得 TIOBE 網頁，列出所有 HTML 表格，再選取第 2 個表格並顯示前 20 筆資料。

## 1. 安裝及載入模組

本範例使用 `requests` 取得 TIOBE 網頁、`BeautifulSoup` 解析 HTML，並用 `pandas` 整理表格。

In [ ]:
!pip -q install requests beautifulsoup4 pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd

## 2. 取得網頁 HTML

先發送 HTTP GET 請求，再檢查回應狀態。

In [ ]:
url = "https://www.tiobe.com/tiobe-index/"

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; ColabWebScraper/1.0)"
}

response = requests.get(url, headers=headers, timeout=20)
response.raise_for_status()

print("HTTP 狀態碼：", response.status_code)

## 3. 找出網頁中的所有表格

`find_all("table")` 會回傳所有 HTML table 標籤。Python 的索引從 0 開始，因此索引值 `1` 代表第 2 個表格。先列出每個表格的索引及資料列數，方便確認目標表格。

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

tables = soup.find_all("table")

print("網頁找到的表格數量：", len(tables))

for i, table in enumerate(tables):
    print(
        "表格索引：", i,
        "資料列數：", len(table.find_all("tr"))
    )

## 4. 讀取第 2 個表格

此處示範選取 `tables[1]`。如果網站改版、表格順序改變，請依上一個步驟的檢查結果調整索引值。

In [ ]:
target_index = 1  # 第2個表格；Python索引從0開始

if len(tables) <= target_index:
    raise IndexError("網頁沒有第 2 個 table，請檢查網站 HTML 或表格數量。")

table = tables[target_index]
rows = table.find_all("tr")

# 讀取第一列欄位名稱
column_names = [
    cell.get_text(" ", strip=True)
    for cell in rows[0].find_all(["th", "td"])
]

# 讀取其餘資料列
data = []
for row in rows[1:]:
    cells = row.find_all(["th", "td"])
    values = [cell.get_text(" ", strip=True) for cell in cells]
    if values:
        data.append(values)

# 補齊欄位數差異，避免建立 DataFrame 時發生錯誤
max_cols = max([len(column_names)] + [len(r) for r in data]) if data else len(column_names)
column_names = (column_names + [f"欄位{i}" for i in range(len(column_names)+1, max_cols+1)])[:max_cols]
data = [r + [""] * (max_cols - len(r)) for r in data]

df = pd.DataFrame(data, columns=column_names)

print("資料筆數：", len(df))
display(df.head(20))

## 5. 也可以用 pandas.read_html() 交叉檢查

`pandas.read_html()` 能直接嘗試讀取網頁中的 HTML 表格。它適合快速擷取表格，但本範例主要目的是練習 `requests` 與 `BeautifulSoup`，因此以上述解析流程為主。

若網站回傳的 HTML 含有可解析的表格，可執行：

In [ ]:
# 可選擇執行：直接讓 pandas 讀取網頁表格
# html_tables = pd.read_html(response.text)
# print("pandas 找到的表格數量：", len(html_tables))
# display(html_tables[1].head(20))

## 6. 程式重點整理

- `soup.find_all("table")`：找出全部表格。
- `tables[1]`：選取第 2 個表格。
- `enumerate(tables)`：逐一列出表格索引及內容。
- `find_all("tr")`：找出表格資料列。
- `find_all(["th", "td"])`：擷取標題格與資料格。

**注意：** TIOBE 網站內容會更新，HTML 表格順序及結構也可能改變。若選到的不是程式語言排名表，請先檢查各表格的列數、欄位或 HTML class，再調整 `target_index`。網站也可能限制自動請求。